## Study Assistant (Project) LangGraph

In [ ]:
# RAG System
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.tools import Tool
from langchain_core.tools.retriever import create_retriever_tool # used for creating local pdf retreival tool


In [ ]:
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.9)

In [ ]:
DATA_PATH = "." # here DATA_PATH = . means we have taken root of folder .
# Directory loader is used to load multiple data or documents like pdf 
loader = DirectoryLoader(DATA_PATH, glob="**/*.pdf", show_progress=True, loader_cls=PyPDFLoader)
documents = loader.load()
print(f"Loaded {len(documents)} total pages.")

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
texts = text_splitter.split_documents(documents)
print(f"Split into {len(texts)} chunks of text (with overlap).")


In [ ]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small", dimensions=1024)
vector_store = FAISS.from_documents(texts, embeddings)
vector_store.save_local("faiss_local_db")
load_retriever = vector_store.as_retriever(search_kwargs={"k": 3}) # This tells the database to return the top 3 most relevant chunks when asked a question



In [ ]:
# Local pdf tool 
local_pdf_tool = create_retriever_tool(
    retriever=vector_store.as_retriever(),
    name="local_pdf_search",
    description="Search the user's uploaded local PDFs. Always use this first if the user asks about their own documents."
)

In [ ]:
# Tavily Search Tool 
# It is used for web search 

from langchain_community.tools.tavily_search import TavilySearchResults

tavily = TavilySearchResults(description="use this tool to get the latest news and information from the web")
tavily.invoke("latest news on AI")

In [ ]:
tools = [local_pdf_tool, tavily]

In [ ]:
## LangGraph Setup 
from typing_extensions import TypedDict
from langchain_core.messages import AnyMessage 
from typing import Annotated # labelling a type with additional information.
from langgraph.graph.message import add_messages # Reducers in langGraph are used to process and transform messages as they flow through the graph.
from IPython.display import Image, display
from langgraph.prebuilt import ToolNode
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import tools_condition
from IPython.display import Image, display
from langgraph.checkpoint.memory import MemorySaver

In [ ]:
class StudyState(TypedDict):
     messages: Annotated[list[AnyMessage], add_messages] # This field will hold the conversation history,
     # and the add_messages reducer will ensure that new messages are appended to this list as they are generated or received. 
     plan: list[str]
     llm_calls : int
     current_step: int

llm_with_tools = llm.bind_tools(tools)

In [ ]:
# 1. Define the Nodes
def planner(state: StudyState):
    prompt = "Create a 3-step Study Plan based on the topic. Return ONLY the 3 sub-topics separated by commas. No intro text."
    response = llm.invoke([("system", prompt)] + state["messages"]) 
    plan_list = [item.strip() for item in response.content.split(",")] # This line takes the response from the LLM, splits it by commas, and then strips any extra whitespace from each item to create a clean list of sub-topics.
    
    return {
        "plan": plan_list, 
        "current_step": 0,
        "messages": [("ai", f"Plan created: {', '.join(plan_list)}. Let's start with {plan_list}!")]
    }

def explainer(state: StudyState):
    current_topic = state["plan"][state["current_step"]] # This line retrieves the current topic that the user is studying based on the current step in the study plan. It accesses the "plan" list from the state and uses the "current_step" index to get the specific topic that needs to be explained.
    prompt = f"Using your tools, explain {current_topic} in detail from the user's notes."
    response = llm_with_tools.invoke([("system", prompt)] + state["messages"]) # This line sends the prompt along with the conversation history to the LLM that has access to the tools.
    #The LLM can then decide to use the tools to gather information and generate a detailed explanation of the current topic.
    return {"messages": [response]}

def quizzer(state: StudyState):
    topic = state["plan"][state["current_step"]]
    prompt = f"The user just learned about {topic}. Create 1 challenging Multiple Choice Question to test their understanding."
    response = llm_with_tools.invoke([("system", prompt)] + state["messages"])
    return {"messages": [response]}

def grader(state: StudyState):
    prompt = """
    You are an expert examiner. Look at the quiz question provided earlier and the user's response.
    1. Determine if the answer is correct.
    2. Provide a brief explanation.
    3. Start with 'CORRECT' or 'INCORRECT'.
    """
    response = llm.invoke([("system", prompt)] + state["messages"])
    is_correct = "CORRECT" in response.content.upper()

    new_step = state["current_step"]
    if is_correct:
        new_step += 1
        
    return {
        "messages": [response],
        "current_step": new_step
    }

def should_continue(state: StudyState): # this checks if there are more topics to study or if we have completed the plan. If the current step index is greater than or equal to the length of the plan, it means we have covered all the topics and can end the session. Otherwise, we go back to the explainer to continue with the next topic.
    if state["current_step"] >= len(state["plan"]):
        return END
    return "explainer"

# 2. Build the Graph
builder = StateGraph(StudyState)

builder.add_node("planner", planner)
builder.add_node("explainer", explainer)
builder.add_node("quizzer", quizzer)
builder.add_node("tools", ToolNode(tools))
builder.add_node("grader", grader)

# Define the Flow
builder.add_edge(START, "planner")
builder.add_edge("planner", "explainer")

# Explainer logic: decides to use tools or go to quizzer
builder.add_conditional_edges(
    "explainer",
    tools_condition,
    {
        "tools": "tools",
        "__end__": "quizzer" 
    }
)

builder.add_edge("tools", "explainer")
builder.add_edge("quizzer", "grader")

# Grader logic: decides to loop back for next topic or finish
builder.add_conditional_edges(
    "grader", 
    should_continue, 
    {
        "explainer": "explainer", 
        END: END
    }
)

# 3. Compile
memory = MemorySaver()
graph = builder.compile(checkpointer=memory, interrupt_before=["grader"])

# 4. Visualize
display(Image(graph.get_graph().draw_mermaid_png()))